# Ingest a snapshot

This notebook runs against the committed OSM fixture
`tests/fixtures/catalog.osm` — five hiking relations, two of them DayHikes.
That is the small subset. It does not download Geofabrik extracts.

`build_catalog` / `dayhike ingest` **stream** the snapshot. They do not load
the whole file into RAM. A continent PBF uses the same stages.

| Stage | What happens |
| --- | --- |
| `filter` | If `osmium` is on `PATH` and the input is PBF, write a hiking-and-amenity extract. XML skips this. |
| `relations` | Keep `route=hiking` relations and amenity relations. |
| `ways` | Keep member ways of those relations (plus amenity ways). |
| `nodes` | Keep coordinates and `ele` only for those members. |
| `access` | Keep driveable highways and parking inside the Trailhead access radius of kept DayHike endpoints. |
| `assemble` | Apply naming and length-band rules; build DayHikes, Trailheads, AmenityFacts. |
| `write` | Write a temp Catalog, then replace `catalog.sqlite`. |

Progress lines go to the `progress` callback (CLI: stderr). Work lands in
`catalog.sqlite.work` next to the Catalog. Re-running the same inputs resumes
that file. `fresh=True` discards it. The previous Catalog stays until `write`
finishes.


In [ ]:
import tempfile
from pathlib import Path

from dayhike import build_catalog

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

work_dir = Path(tempfile.mkdtemp())
print("fixture", fixture)
print("work_dir", work_dir)


## Progress callback

Each line starts with the stage name. On the fixture this is instant; on a
Geofabrik extract the same names appear for minutes or hours.


In [ ]:
catalog_path = work_dir / "catalog.sqlite"
messages = []

def show(message: str) -> None:
    messages.append(message)
    print(message)

catalog = build_catalog(fixture, catalog_path, progress=show)
print(catalog.ingest_report)
print("sqlite", catalog.path)
print("work leftover", catalog_path.with_name(catalog_path.name + ".work").exists())
for hike in catalog.dayhikes:
    print(hike.id, hike.name, round(hike.length_m))


## Resume after a kill

If ingest stops mid-run, the Catalog is not replaced. The work file remains.
The next `build_catalog` on the same inputs continues. Completed stages print
`resumed, already complete`.


In [ ]:
resume_path = work_dir / "resume.sqlite"
work_path = resume_path.with_name(resume_path.name + ".work")

def kill_on_access(message: str) -> None:
    print(message)
    if message.startswith("access:"):
        raise RuntimeError("killed")

try:
    build_catalog(fixture, resume_path, progress=kill_on_access)
except RuntimeError as exc:
    print("interrupted", exc)

print("catalog exists", resume_path.exists())
print("work exists", work_path.exists())

resumed = build_catalog(fixture, resume_path, progress=print)
print(resumed.ingest_report)
print("Ridge Walk" in {hike.name for hike in resumed.dayhikes})
print("Valley Path" in {hike.name for hike in resumed.dayhikes})
print("work leftover", work_path.exists())


## Unused objects are skipped

A residential way far from the fixture DayHikes is not a hiking route. Ingest
still keeps Ridge Walk and Valley Path. On a continent extract this is the
same idea: buildings and unrelated roads never become DayHikes and are not
held in RAM.


In [ ]:
noisy = work_dir / "noisy.osm"
extra = "".join(
    f'  <node id="{9000 + i}" lat="46.0" lon="16.0" version="1"/>\n'
    for i in range(20)
)
extra += """  <way id="9001" version="1">
    <nd ref="9000"/>
    <nd ref="9019"/>
    <tag k="highway" v="residential"/>
  </way>
</osm>
"""
noisy.write_text(fixture.read_text().replace("</osm>", extra))
noisy_catalog = build_catalog(noisy, work_dir / "noisy.sqlite", progress=print)
print(noisy_catalog.ingest_report)
print([hike.name for hike in noisy_catalog.dayhikes])


## CLI

```bash
uv run dayhike ingest tests/fixtures/catalog.osm --catalog data/catalog.sqlite
uv run dayhike ingest tests/fixtures/catalog.osm --catalog data/catalog.sqlite --list
uv run dayhike ingest data/osm/europe-latest.osm.pbf --catalog data/catalog.sqlite
uv run dayhike ingest data/osm/europe-latest.osm.pbf --catalog data/catalog.sqlite --fresh
```

Stdout: path and `IngestReport`. Stderr: stage lines. `--list` prints kept
DayHikes. `--fresh` ignores `catalog.sqlite.work`.

If osmium-tool is installed, PBF ingest writes `*.catalog.osm.pbf` next to the
source (hiking relations, huts, parking, cafes, drinking water, plus referenced
members). DriveTime still needs the unfiltered extract and
`data/valhalla/prepare-roads.sh` — do not feed the hiking extract to Valhalla.
